# TravelMate AI — Giai đoạn 4: Demo suy luận

Notebook nạp Qwen3-4B cùng LoRA adapter và gửi một câu hỏi du lịch. Bản demo ứng dụng trên laptop vẫn chạy qua FastAPI/Expo Web; notebook này dùng để kiểm tra riêng mô hình.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/trongnd16092005/travelmate-ai.git"
BRANCH = "feature/ai-itinerary-generation"
REPO_DIR = Path("/content/travelmate-ai")
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR / "services" / "ai-service")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[training]"], check=True)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3-4B"
ADAPTER_PATH = Path("/content/drive/MyDrive/TravelMate/artifacts/travelmate-qwen3-4b-lora")
if not torch.cuda.is_available():
    raise RuntimeError("Hãy chọn GPU runtime trong Colab.")
if not ADAPTER_PATH.exists():
    raise FileNotFoundError(f"Không tìm thấy adapter tại {ADAPTER_PATH}.")

use_bf16 = torch.cuda.is_bf16_supported()
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
quantization = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, device_map="auto", torch_dtype=compute_dtype, quantization_config=quantization
)
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH).eval()
print("Đã nạp model và adapter.")

In [ ]:
from app.prompts.chat import CHAT_SYSTEM_PROMPT
from training.generate_predictions import generate_response

QUESTION = "Lập lịch trình Đà Nẵng 3 ngày cho 2 người với ngân sách 5 triệu."
messages = [
    {"role": "system", "content": CHAT_SYSTEM_PROMPT},
    {"role": "user", "content": QUESTION},
]
response = generate_response(model, tokenizer, messages, max_new_tokens=512)
print("Người dùng:", QUESTION)
print("\nTravelMate:", response)